## Answer the questions that decide the model design.

In [2]:
import json
import pandas as pd

In [5]:
RAW = "../data/raw"

tr = pd.read_csv(f"{RAW}/train.csv", parse_dates=["timestamp"])
vi = pd.read_csv(f"{RAW}/validation_input.csv", parse_dates=["timestamp"])
fi = pd.read_csv(f"{RAW}/forecast_index_validation.csv", parse_dates=["timestamp"])
meta = json.load(open(f"{RAW}/metadata.json"))

In [6]:
def rule(t): print("\n" + "=" * 62 + f"\n{t}\n" + "=" * 62)

#### 1. shapes and coverage

In [7]:
rule("1. SHAPES")
for name, d in [("train", tr), ("val_input", vi), ("fcst_index", fi)]:
    per = d.groupby("series_id").size()
    print(f"{name:11s} {d.shape}  series={d.series_id.nunique():3d}  "
          f"rows/series={per.min()}..{per.max()}")
    print(f"{'':11s} {d.timestamp.min()} -> {d.timestamp.max()}")


1. SHAPES
train       (414720, 25)  series= 96  rows/series=4320..4320
            2023-01-01 00:00:00 -> 2023-06-29 23:00:00
val_input   (32256, 24)  series= 96  rows/series=336..336
            2023-06-30 00:00:00 -> 2023-07-13 23:00:00
fcst_index  (32256, 2)  series= 96  rows/series=336..336
            2023-06-30 00:00:00 -> 2023-07-13 23:00:00


#### 2. are future covariates given?

In [8]:
rule("2. DO WE GET FEATURES FOR THE FORECAST WINDOW?")
fkeys = set(map(tuple, fi[["series_id", "timestamp"]].values))
vkeys = set(map(tuple, vi[["series_id", "timestamp"]].values))
covered = len(fkeys & vkeys)
print(f"forecast rows            : {len(fkeys)}")
print(f"of those, present in vi  : {covered}")
print(f"=> FUTURE COVARIATES     : {'YES' if covered == len(fkeys) else 'NO'}")
print(f"\nvalidation_input has target column: {'target' in vi.columns}")
if "target" in vi.columns:
    print(f"  ... non-null targets in it: {vi.target.notna().sum()}")


2. DO WE GET FEATURES FOR THE FORECAST WINDOW?
forecast rows            : 32256
of those, present in vi  : 32256
=> FUTURE COVARIATES     : YES

validation_input has target column: False


#### 3. the gap between train end and forecast start

In [9]:
rule("3. TIMELINE")
gap = (fi.timestamp.min() - tr.timestamp.max()).total_seconds() / 3600
print(f"train ends        {tr.timestamp.max()}")
print(f"forecast starts   {fi.timestamp.min()}")
print(f"gap               {gap:.0f} hours")
print(f"forecast length   {fi.groupby('series_id').size().iloc[0]} hours")


3. TIMELINE
train ends        2023-06-29 23:00:00
forecast starts   2023-06-30 00:00:00
gap               1 hours
forecast length   336 hours


#### 4. missing values

In [10]:
rule("4. MISSING VALUES (train)")
miss = tr.isna().mean().sort_values(ascending=False)
print(miss[miss > 0].round(4).to_string() or "none")


4. MISSING VALUES (train)
shock_risk                             0.0456
upstream_quality_forecast              0.0453
event_load_forecast                    0.0453
service_irregularity_risk_forecast     0.0451
network_pressure_forecast              0.0451
staffing_forecast                      0.0449
throughput_disruption_risk_forecast    0.0449
demand_forecast                        0.0448
queue_pressure_forecast                0.0447
unit_reliability_forecast              0.0446


#### 5. which features never change inside a series?

In [11]:
rule("5. STATIC vs TIME-VARYING")
feats = [c for c in tr.columns if c not in ("series_id", "timestamp", "target")]
nun = tr.groupby("series_id")[feats].nunique().max()
for c in feats:
    print(f"  {'STATIC     ' if nun[c] == 1 else 'time-varying'} {c}")


5. STATIC vs TIME-VARYING
  time-varying hour_sin
  time-varying hour_cos
  time-varying dow_sin
  time-varying dow_cos
  time-varying is_weekend
  time-varying trend
  time-varying workload_intensity
  time-varying demand_forecast
  time-varying staffing_forecast
  time-varying upstream_quality_forecast
  time-varying promotion_intensity
  time-varying shock_risk
  time-varying maintenance_known
  time-varying unit_reliability_forecast
  time-varying queue_pressure_forecast
  time-varying network_pressure_forecast
  time-varying event_load_forecast
  time-varying service_irregularity_risk_forecast
  time-varying throughput_disruption_risk_forecast
  STATIC      nominal_capacity
  STATIC      zone_sin
  STATIC      zone_cos


#### 6. target

In [13]:
rule("6. TARGET")
print(tr.target.describe().round(3).to_string())
pm = tr.groupby("series_id").target.mean()
print(f"\nper-series mean: min={pm.min():.2f}  max={pm.max():.2f}  "
      f"ratio={pm.max()/pm.min():.1f}x")
print(f"target NaNs    : {tr.target.isna().sum()}")


6. TARGET
count    414720.000
mean          9.913
std           5.548
min           0.164
25%           5.801
50%           9.042
75%          13.026
max          53.000

per-series mean: min=6.45  max=14.38  ratio=2.2x
target NaNs    : 0


In [14]:
rule("metadata.json")
print(json.dumps(meta, indent=2)[:1200])


metadata.json
{
  "name": "operations_forecasting_2026",
  "task": "Multivariate hourly forecasting for anonymized operations units.",
  "target_column": "target",
  "target_description": "Predict the future hourly operational load index for each series_id. Higher values indicate more operational pressure in that unit.",
  "frequency": "h",
  "n_series": 96,
  "n_steps": 4992,
  "history_length": 168,
  "forecast_horizon": 24,
  "forecast_horizon_note": "Rollout block length. Submissions must still predict every row in the provided forecast_index_*.csv files.",
  "validation_horizon": 336,
  "test_horizon": 336,
  "required_prediction_horizon": {
    "validation": 336,
    "test": 336
  },
  "metric": "wape",
  "schema": {
    "train": [
      "series_id",
      "timestamp",
      "hour_sin",
      "hour_cos",
      "dow_sin",
      "dow_cos",
      "is_weekend",
      "trend",
      "workload_intensity",
      "demand_forecast",
      "staffing_forecast",
      "upstream_quality_fore